# knowledge graph

### step 1: nodes, edges and triples
#### TRIPLE-> every fact, irrespective of its complexity can be broken down into 3 parts: HEAD, RELATION, TAIL

In [ ]:
import os

In [ ]:
from dotenv import load_dotenv
 

In [ ]:
load_dotenv()

In [ ]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [ ]:
GROQ_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")

In [ ]:

NEO4J_URI = os.getenv("NEO4J_URI") or None
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

In [ ]:
from collections import Counter

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import networkx as nx

### the following snippet is in the form of dictionary where the entity names are represented as keys and and values represent the additional info such as their labels andthe year in which movie was released

In [ ]:
NODES={
    "Christopher Nolan":   {"label":"Person"},
    "Leonardo DiCaprio" :  {"label":"Person"},
    "Tom Hardy":           {"label":"Person"},
    "Matthew McConaughey": {"label": "Person"},
    "James Cameron":       {"label": "Person"},
    "Kate Winslet":        {"label": "Person"},
    "Lana Wachowski":      {"label": "Person"},
    "Lilly Wachowski":     {"label": "Person"},
    "Keanu Reeves":        {"label": "Person"},
    "Inception":           {"label": "Movie", "year": 2010},
    "Interstellar":        {"label": "Movie", "year": 2014},
    "Titanic":             {"label": "Movie", "year": 1997},
    "The Matrix":          {"label": "Movie", "year": 1999},
    "Sci-Fi":              {"label": "Genre"},
    "Action":              {"label": "Genre"},
    "Drama":               {"label": "Genre"},
    "Riya":                {"label": "User"},   # private data: a movie club
    "Kabir":               {"label": "User"},
}





#### we use list as it allows us to store multiple triples together, inside the list the triples are stored in for of tuples to store data sequentially , there are 4 elements present in a tuple, the three are SPO(subject, predicate,object) and the last one is for additional relationship attributes

In [ ]:
TRIPLES=[
     ("Christopher Nolan", "DIRECTED", "Inception", {}),
    ("Leonardo DiCaprio", "ACTED_IN", "Inception", {"role": "Cobb"}),
    ("Tom Hardy", "ACTED_IN", "Inception", {"role": "Eames"}),
    ("Inception", "HAS_GENRE", "Sci-Fi", {}),
    ("Inception", "HAS_GENRE", "Action", {}),
 
    ("Christopher Nolan", "DIRECTED", "Interstellar", {}),
    ("Matthew McConaughey", "ACTED_IN", "Interstellar", {"role": "Cooper"}),
    ("Interstellar", "HAS_GENRE", "Sci-Fi", {}),
    ("Interstellar", "HAS_GENRE", "Drama", {}),
 
    ("James Cameron", "DIRECTED", "Titanic", {}),
    ("Leonardo DiCaprio", "ACTED_IN", "Titanic", {"role": "Jack Dawson"}),
    ("Kate Winslet", "ACTED_IN", "Titanic", {"role": "Rose"}),
    ("Titanic", "HAS_GENRE", "Drama", {}),
 
    ("Lana Wachowski", "DIRECTED", "The Matrix", {}),
    ("Lilly Wachowski", "DIRECTED", "The Matrix", {}),
    ("Keanu Reeves", "ACTED_IN", "The Matrix", {"role": "Neo"}),
    ("The Matrix", "HAS_GENRE", "Sci-Fi", {}),
    ("The Matrix", "HAS_GENRE", "Action", {}),
 
    ("Riya", "WATCHED", "Inception", {"rating": 5}),
    ("Riya", "WATCHED", "The Matrix", {"rating": 4}),
    ("Riya", "WATCHED", "Titanic", {"rating": 2}),
    ("Kabir", "WATCHED", "Interstellar", {"rating": 5}),
]

In [ ]:
SCHEMA = {
    "DIRECTED":  ("Person", "Movie"),
    "ACTED_IN":  ("Person", "Movie"),
    "HAS_GENRE": ("Movie", "Genre"),
    "WATCHED":   ("User", "Movie"),
}

In [ ]:
def build_graph():
    G = nx.MultiDiGraph()
    for name, attrs in NODES.items():
        G.add_node(name, **attrs)
 
    for head, rel, tail, props in TRIPLES:
        if rel not in SCHEMA:
            raise ValueError(f"Unknown relationship: {rel}")
        want_head, want_tail = SCHEMA[rel]
        got_head, got_tail = G.nodes[head]["label"], G.nodes[tail]["label"]
        if (got_head, got_tail) != (want_head, want_tail):
            raise ValueError(f"{rel} must go {want_head}->{want_tail}, got {got_head}->{got_tail}")
        # key=rel means adding the same fact twice doesn't create a duplicate edge
        G.add_edge(head, tail, key=rel, relation=rel, **props)
 
    return G

In [ ]:
def outgoing(G, node, rel):
    """Nodes this node points TO through `rel`."""
    return [t for _, t, k in G.out_edges(node, keys=True) if k == rel]
 
def incoming(G, node, rel):
    """Nodes that point AT this node through `rel`."""
    return [h for h, _, k in G.in_edges(node, keys=True) if k == rel]
 
# 1 hop:  (person)-[DIRECTED]->(movie)
def movies_directed_by(G, person):
    return sorted(outgoing(G, person, "DIRECTED"), key=lambda m: G.nodes[m]["year"])
 
# 1 hop, reading a property stored on the edge
def cast_of(G, movie):
    return [(actor, G.edges[actor, movie, "ACTED_IN"]["role"]) for actor in incoming(G, movie, "ACTED_IN")]

In [ ]:
def co_actors(G, actor):
    result = {}
    for movie in outgoing(G, actor, "ACTED_IN"):
        for other in incoming(G, movie, "ACTED_IN"):
            if other != actor:
                result.setdefault(other, []).append(movie)
    return result

In [ ]:
def unwatched_by(G, user, director):
    seen = set(outgoing(G, user, "WATCHED"))
    return [m for m in movies_directed_by(G, director) if m not in seen]
 
# Path: how are two things connected? (direction ignored; some labels can be skipped)
def connection(G, a, b, avoid_labels=()):
    U = G.to_undirected()
    U.remove_nodes_from([n for n, d in G.nodes(data=True) if d["label"] in avoid_labels and n not in (a, b)])
    try:
        return nx.shortest_path(U, a, b)
    except nx.NetworkXNoPath:
        return None

In [ ]:
COLORS = {"Person": "#4C9BE8", "Movie": "#F2A541", "Genre": "#8BC34A", "User": "#E35D6A"}

In [ ]:
def draw(G):
    pos = nx.spring_layout(G, k=1.2, seed=7)
    plt.figure(figsize=(13, 9))
    nx.draw_networkx_nodes(G, pos, node_size=1100,
                           node_color=[COLORS[G.nodes[n]["label"]] for n in G])
    nx.draw_networkx_labels(G, pos, font_size=8)
    nx.draw_networkx_edges(G, pos, arrows=True, arrowsize=14, alpha=0.5)
    nx.draw_networkx_edge_labels(G, pos, font_size=6,
                                 edge_labels={(u, v): k for u, v, k in G.edges(keys=True)})
    for label, color in COLORS.items():
        plt.scatter([], [], c=color, s=100, label=label)
    plt.legend(loc="upper left")
    plt.title("Movie knowledge graph")
    plt.axis("off")
    plt.show()

In [ ]:
if __name__ == "__main__":
    G = build_graph()
    print(f"{G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    print("By label:", dict(Counter(d["label"] for _, d in G.nodes(data=True))))
 
    print("\nAll facts:")
    for head, tail, data in G.edges(data=True):
        props = {k: v for k, v in data.items() if k != "relation"}
        print(f"  ({head}) -[{data['relation']}{' ' + str(props) if props else ''}]-> ({tail})")
 
    print("\nQueries:")
    print("  Nolan directed:          ", movies_directed_by(G, "Christopher Nolan"))
    print("  Cast of Inception:       ", cast_of(G, "Inception"))
    print("  DiCaprio's co-actors:    ", co_actors(G, "Leonardo DiCaprio"))
    print("  Nolan films Riya missed: ", unwatched_by(G, "Riya", "Christopher Nolan"))
    print("  Tom Hardy -> Kate Winslet:", connection(G, "Tom Hardy", "Kate Winslet"))
    print("  DiCaprio -> Keanu:        ", connection(G, "Leonardo DiCaprio", "Keanu Reeves"))
    print("  ...avoiding Genre/User:   ", connection(G, "Leonardo DiCaprio", "Keanu Reeves",
                                                    avoid_labels=("Genre", "User")))
 
    draw(G)
 

In [ ]:
print(SCHEMA)